# Clinical Trial Criteria Extraction & Grounding Demo

This notebook demonstrates the ElixirTrials pipeline end-to-end:
1. **PDF Ingestion** — Load a clinical trial protocol PDF
2. **Criteria Extraction** — Extract structured eligibility criteria via Gemini
3. **Rich Display** — Visualize criteria with type badges, confidence, thresholds
4. **Entity Grounding** — Ground medical entities to standard terminologies
5. **Expression Trees** — Visualize logical structure of compound criteria
6. **Field Mappings** — Display Entity-Relation-Value-Unit decomposition

**Launch with:** `uv run jupyter notebook notebooks/criteria_extraction_demo.ipynb`

## 1. Setup

Load environment variables and add workspace source paths.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add workspace source paths
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for src_path in [
    ROOT / "services" / "protocol-processor-service" / "src",
    ROOT / "services" / "api-service" / "src",
    ROOT / "libs" / "inference" / "src",
    ROOT / "libs" / "shared" / "src",
    ROOT / "libs" / "events-py" / "src",
]:
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

# Load environment variables
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT / ".env.local", override=True)

api_key = os.getenv("GOOGLE_API_KEY")
backend = os.getenv("MODEL_BACKEND", "gemini")
print(f"Backend: {backend}")
print(f"GOOGLE_API_KEY: {'set' if api_key else 'NOT SET'}")
print(f"OLLAMA_BASE_URL: {os.getenv('OLLAMA_BASE_URL', 'not set')}")
print(f"\nWorkspace root: {ROOT}")

## 2. PDF Ingestion

Load a sample protocol PDF, display page count and metadata.

In [ ]:
import pymupdf

# Find sample PDFs
sample_dir = ROOT / "notebooks" / "sample_data"
pdf_files = list(sample_dir.glob("*.pdf"))

if not pdf_files:
    print("No PDF files found in notebooks/sample_data/")
    print("Please place a clinical trial protocol PDF in that directory.")
    print("\nUsing a placeholder for demonstration...")
    pdf_path = None
    pdf_bytes = None
else:
    pdf_path = pdf_files[0]
    pdf_bytes = pdf_path.read_bytes()
    doc = pymupdf.open(str(pdf_path))

    print(f"PDF: {pdf_path.name}")
    print(f"Pages: {len(doc)}")
    print(f"Size: {len(pdf_bytes) / 1024:.1f} KB")

    metadata = doc.metadata
    if metadata:
        print("\nMetadata:")
        for key, val in metadata.items():
            if val:
                print(f"  {key}: {val}")

    # Show first page preview
    first_page = doc[0]
    text_preview = first_page.get_text()[:500]
    print(f"\n--- First page preview ---\n{text_preview}...")
    doc.close()

## 3. Criteria Extraction

Extract structured eligibility criteria using `extract_criteria_structured()`.
This calls Gemini (or local model via gateway) to parse the PDF into structured criteria.

In [ ]:
import pandas as pd
from protocol_processor.tools.gemini_extractor import extract_criteria_structured
from protocol_processor.schemas.extraction import ExtractionResult

if pdf_bytes:
    # Run extraction
    result_json = await extract_criteria_structured(
        pdf_bytes=pdf_bytes,
        protocol_id="demo-001",
        title=pdf_path.stem if pdf_path else "Demo Protocol",
    )
    extraction = ExtractionResult.model_validate_json(result_json)

    print(f"Extracted {len(extraction.criteria)} criteria")
    print(f"Protocol summary: {extraction.protocol_summary[:200]}...")

    # Display as DataFrame
    rows = []
    for c in extraction.criteria:
        rows.append(
            {
                "Type": c.criteria_type,
                "Category": c.category or "—",
                "Text": c.text[:100] + ("..." if len(c.text) > 100 else ""),
                "Confidence": f"{c.confidence:.2f}",
                "Assertion": c.assertion_status or "—",
                "Page": c.page_number or "—",
            }
        )

    df = pd.DataFrame(rows)
    display(df)
else:
    print("Skipping extraction — no PDF loaded.")
    extraction = None

## 4. Rich Criteria Display

Visualize criteria with type badges, temporal constraints, numeric thresholds, and confidence histogram.

In [ ]:
import matplotlib.pyplot as plt

if extraction and extraction.criteria:
    criteria = extraction.criteria

    # Type badge colors
    TYPE_COLORS = {"inclusion": "#22c55e", "exclusion": "#ef4444"}

    # Display each criterion with rich formatting
    for i, c in enumerate(criteria[:10]):  # Show first 10
        color = TYPE_COLORS.get(c.criteria_type, "#6b7280")
        badge = f"\033[48;2;{int(color[1:3], 16)};{int(color[3:5], 16)};{int(color[5:7], 16)}m"

        print(f"\n{'=' * 70}")
        print(
            f"[{c.criteria_type.upper()}] [{c.category or 'uncategorized'}] (confidence: {c.confidence:.2f})"
        )
        print(f"  {c.text[:200]}")

        if c.temporal_constraint:
            tc = c.temporal_constraint
            print(f"  Temporal: {tc.duration} {tc.relation} {tc.reference_point or ''}")

        if c.numeric_thresholds:
            for nt in c.numeric_thresholds:
                upper = f"–{nt.upper_value}" if nt.upper_value else ""
                print(f"  Threshold: {nt.comparator} {nt.value}{upper} {nt.unit or ''}")

    # Confidence histogram
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    confidences = [c.confidence for c in criteria]
    axes[0].hist(confidences, bins=20, color="#3b82f6", edgecolor="white", alpha=0.8)
    axes[0].set_xlabel("Confidence")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Confidence Distribution")
    axes[0].axvline(x=0.7, color="#f59e0b", linestyle="--", label="Threshold (0.7)")
    axes[0].legend()

    # Type breakdown
    type_counts = {}
    for c in criteria:
        type_counts[c.criteria_type] = type_counts.get(c.criteria_type, 0) + 1
    colors = [TYPE_COLORS.get(t, "#6b7280") for t in type_counts.keys()]
    axes[1].bar(type_counts.keys(), type_counts.values(), color=colors)
    axes[1].set_title("Criteria by Type")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.show()
else:
    print("No criteria to display.")

## 5. Entity Grounding

For selected criteria, use `TerminologyRouter.route_entity()` for candidates,
then `medgemma_decide()` for selection. Displays `EntityGroundingResult` with candidate comparison.

In [ ]:
from protocol_processor.tools.terminology_router import TerminologyRouter

if extraction and extraction.criteria:
    router = TerminologyRouter()

    # Pick a criterion with conditions/entities
    sample_criterion = extraction.criteria[0]
    print(f"Grounding criterion: {sample_criterion.text[:150]}...")
    print(f"Category: {sample_criterion.category}")
    print()

    # Extract entity terms from the criterion
    # In the full pipeline, entities are extracted during the grounding phase
    # Here we demonstrate the routing for a sample entity
    sample_entity = {
        "text": sample_criterion.text.split()[0:3],  # First few words as entity
        "entity_type": sample_criterion.category or "condition",
    }
    entity_text = " ".join(sample_entity["text"])

    print(f"Routing entity: '{entity_text}' (type: {sample_entity['entity_type']})")

    try:
        candidates = await router.route_entity(
            entity_text=entity_text,
            entity_type=sample_entity["entity_type"],
        )

        if candidates:
            print(f"\nFound {len(candidates)} candidates:\n")
            candidate_rows = []
            for cand in candidates[:10]:
                candidate_rows.append(
                    {
                        "Source": cand.source_api,
                        "Code": cand.code,
                        "Term": cand.preferred_term[:60],
                        "Score": f"{cand.score:.3f}",
                        "Type": cand.semantic_type or "—",
                    }
                )
            display(pd.DataFrame(candidate_rows))

            # Try MedGemma decision
            try:
                from protocol_processor.tools.medgemma_decider import medgemma_decide

                result = await medgemma_decide(
                    entity={
                        "text": entity_text,
                        "entity_type": sample_entity["entity_type"],
                    },
                    candidates=candidates,
                    criterion_context=sample_criterion.text,
                )
                print("\nMedGemma Decision:")
                print(f"  Selected: {result.selected_code} ({result.selected_system})")
                print(f"  Term: {result.preferred_term}")
                print(f"  Confidence: {result.confidence:.2f}")
                print(f"  Reasoning: {result.reasoning[:200]}")
            except Exception as e:
                print(
                    f"\nMedGemma unavailable ({e.__class__.__name__}): showing candidates only."
                )
                print("(This is expected if MedGemma/Vertex is not configured)")
        else:
            print("No grounding candidates found.")
    except Exception as e:
        print(f"Grounding failed: {e}")
        print("(This may happen if ToolUniverse API is not accessible)")
else:
    print("No criteria available for grounding.")

## 6. Expression Tree Visualization

Visualize `StructuredCriterionTree` as an indented text tree with unicode box-drawing characters.

In [ ]:
from protocol_processor.schemas.structure import ExpressionNode, StructuredCriterionTree


def render_tree(
    node: dict | ExpressionNode, prefix: str = "", is_last: bool = True
) -> str:
    """Render an expression tree as indented text with box-drawing chars."""
    if isinstance(node, dict):
        node = ExpressionNode.model_validate(node)

    connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
    lines = []

    if node.type == "ATOMIC":
        entity = node.entity or "?"
        relation = node.relation or ""
        value = node.value or ""
        unit = node.unit or ""
        label = f"\u25cf {entity} {relation} {value} {unit}".strip()
    else:
        label = f"\u25b6 {node.type}"

    lines.append(f"{prefix}{connector}{label}")

    if node.children:
        child_prefix = prefix + ("    " if is_last else "\u2502   ")
        for i, child in enumerate(node.children):
            is_child_last = i == len(node.children) - 1
            lines.append(render_tree(child, child_prefix, is_child_last))

    return "\n".join(lines)


# Demo with a sample expression tree
sample_tree = StructuredCriterionTree(
    root=ExpressionNode(
        type="AND",
        children=[
            ExpressionNode(
                type="ATOMIC", entity="Age", relation=">=", value="18", unit="years"
            ),
            ExpressionNode(
                type="OR",
                children=[
                    ExpressionNode(
                        type="ATOMIC",
                        entity="HbA1c",
                        relation=">=",
                        value="7.0",
                        unit="%",
                    ),
                    ExpressionNode(
                        type="ATOMIC",
                        entity="Fasting glucose",
                        relation=">=",
                        value="126",
                        unit="mg/dL",
                    ),
                ],
            ),
            ExpressionNode(
                type="NOT",
                children=[
                    ExpressionNode(
                        type="ATOMIC",
                        entity="Pregnancy",
                        relation="contains",
                        value="current",
                    ),
                ],
            ),
        ],
    ),
    structure_confidence="llm",
    structure_model="gemini-2.5-flash",
)

print("Expression Tree (sample):")
print()
print(render_tree(sample_tree.root))
print(f"\nConfidence: {sample_tree.structure_confidence}")
print(f"Model: {sample_tree.structure_model}")

# If extraction produced structured_criterion, show those too
if extraction:
    for c in extraction.criteria[:3]:
        if hasattr(c, "conditions") and c.conditions:
            conds = c.conditions
            if isinstance(conds, list):
                print("\n--- Criterion conditions ---")
                print(f"Text: {c.text[:100]}")
                for cond in conds:
                    print(f"  - {cond}")

## 7. Field Mappings

Display Entity | Relation | Value | Unit tables for the decomposed criteria.

In [ ]:
# Sample field mappings (as would be generated by field_mapper.py)
sample_mappings = [
    {
        "entity": "HbA1c",
        "entity_code": "4548-4",
        "entity_system": "LOINC",
        "relation": ">=",
        "value": "7.0",
        "unit": "%",
        "omop_concept_id": "3004410",
    },
    {
        "entity": "Age",
        "entity_code": "30525-0",
        "entity_system": "LOINC",
        "relation": ">=",
        "value": "18",
        "unit": "years",
        "omop_concept_id": "4265453",
    },
    {
        "entity": "Type 2 Diabetes",
        "entity_code": "E11",
        "entity_system": "ICD-10",
        "relation": "contains",
        "value": "diagnosis",
        "unit": "",
        "omop_concept_id": "201826",
    },
    {
        "entity": "Metformin",
        "entity_code": "6809",
        "entity_system": "RxNorm",
        "relation": ">=",
        "value": "500",
        "unit": "mg/day",
        "omop_concept_id": "1503297",
    },
    {
        "entity": "eGFR",
        "entity_code": "62238-1",
        "entity_system": "LOINC",
        "relation": ">=",
        "value": "30",
        "unit": "mL/min/1.73m2",
        "omop_concept_id": "46236952",
    },
]

print("Field Mappings (Entity-Relation-Value-Unit):\n")

mapping_df = pd.DataFrame(
    [
        {
            "Entity": m["entity"],
            "System": m["entity_system"],
            "Code": m["entity_code"],
            "Relation": m["relation"],
            "Value": m["value"],
            "Unit": m["unit"] or "—",
            "OMOP ID": m.get("omop_concept_id", "—"),
        }
        for m in sample_mappings
    ]
)


# Style the DataFrame
def style_system(val):
    colors = {
        "LOINC": "background-color: #f3e8ff; color: #7c3aed",
        "ICD-10": "background-color: #fff7ed; color: #ea580c",
        "RxNorm": "background-color: #eff6ff; color: #2563eb",
        "SNOMED": "background-color: #f0fdf4; color: #16a34a",
    }
    return colors.get(val, "")


styled = mapping_df.style.map(style_system, subset=["System"])
display(styled)

# If we have actual extraction results with field mappings, show those
if extraction:
    actual_mappings = []
    for c in extraction.criteria:
        if c.conditions and isinstance(c.conditions, list):
            for cond in c.conditions:
                if isinstance(cond, str):
                    actual_mappings.append(
                        {"criterion": c.text[:60], "condition": cond}
                    )

    if actual_mappings:
        print(f"\n\nActual extracted conditions ({len(actual_mappings)} total):")
        display(pd.DataFrame(actual_mappings[:20]))

---

## Summary

This notebook demonstrated the core ElixirTrials pipeline:

| Step | Tool | Output |
|------|------|--------|
| PDF Ingestion | `pymupdf` | Page count, metadata, text |
| Extraction | `extract_criteria_structured()` | `ExtractionResult` with typed criteria |
| Grounding | `TerminologyRouter` + `medgemma_decide()` | `EntityGroundingResult` with codes |
| Structure | Expression tree builder | `StructuredCriterionTree` (AND/OR/NOT) |
| Mapping | Field mapper | Entity-Relation-Value-Unit tables |

**Next steps:**
- Review criteria in the HITL UI (`/criteria` spreadsheet view)
- Export structured criteria for downstream CDM integration
- Run evaluation metrics against gold-standard annotations